# Hodgkin–Huxley Modelling: Hands-On Workshop
University of Melbourne Neuroscience PhD Programme, 16 November 2026

This workshop introduces the NEURON simulation environment in Python.

## Activities
1. **Single action potentials:** gating variables, conductances, and the threshold current
2. **Tonic firing:** long current pulses
3. **Altering conductances:** the effect of the Na⁺ conductance
4. **Action potential generation and propagation** in a multi-compartment neuron
5. **Axonal conduction:** conduction velocity and axon diameter

## Setup
Run these two cells first, in order. The first installs NEURON and compiles the ion-channel mechanisms; the second imports NEURON and the workshop's helper functions (in `sheasbyfohlmeister1999_utils.py`), so it must come after.

If anything doesnt work, click **Runtime → Restart session** and run again from the top.

In [ ]:
#@title ▶ Run this first: installs NEURON and fetches the workshop files (~1 min) { display-mode: "form" }
REPO_URL = "https://github.com/jseo977/UniMelb-PhD-Neuroscience-Computational-Modelling-Workshop.git"

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install NEURON and interactive-plot support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuron==9.0.2", "ipympl"], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()

    # Colab loads matplotlib before ipympl is installed, so register the new
    # 'widget' backend with matplotlib (otherwise %matplotlib widget fails)
    try:
        from matplotlib.backends import backend_registry
        new = [(n, m) for n, m in backend_registry._read_entry_points()
               if n.lower() not in backend_registry._backend_to_gui_framework]
        backend_registry._validate_and_store_entry_points(new)
    except Exception:
        pass   # plots will fall back to static images

    # Fetch the workshop files (.mod, morphology, utils) and move into that folder
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)

# Compile the .mod files (skipped if already compiled; delete the x86_64/arm64 folder to force a rebuild)
if not any(os.path.isdir(d) for d in ("x86_64", "arm64", "aarch64")):
    build = subprocess.run(["nrnivmodl"], capture_output=True, text=True)
    if build.returncode != 0:
        print(build.stdout[-3000:], build.stderr[-3000:])
        raise RuntimeError("Compiling the .mod files failed (see messages above)")

print("Setup complete. Working directory:", os.getcwd())

# Imports and settings used throughout the notebook (run after the setup cell)
import sys
if "google.colab" in sys.modules:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')   # interactive figures (zoom / pan / reset)
    except Exception:
        get_ipython().run_line_magic('matplotlib', 'inline')
        print("Interactive plots unavailable; using static plots.")
else:
    get_ipython().run_line_magic('matplotlib', 'inline')       # static figures (work in VS Code and Jupyter)

import numpy as np
import matplotlib.pyplot as plt
from neuron import h, gui

import importlib, sheasbyfohlmeister1999_utils
importlib.reload(sheasbyfohlmeister1999_utils)   # pick up changes to the utils file without restarting
from sheasbyfohlmeister1999_utils import (
    scale_surface_area, show_single_compartment, report_tonic_firing, report_conductances,    # Activities 1–3
    setup_rgc, show_rgc, rgc_dashboard, rgc_zoom_dashboard, conduction_velocity,              # Activity 4
    show_axon_conduction,                                                                     # Activity 5
)

plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

## The single-compartment model

In [ ]:
# Activity 1, 2, 3

def build_soma_single_compartment():
    """A single cylindrical compartment the size of the soma of ctt3219b, with Hodgkin–Huxley
    Na and delayed-rectifier K channels and a leak. Other ion channels were removed for simplicity."""
    soma_single_compartment = h.Section(name='soma_single_compartment')
    soma_single_compartment.L    = 27.69     # µm (gives the same membrane area as the soma, ~1216 µm²)
    soma_single_compartment.diam = 13.98     # µm
    soma_single_compartment.nseg = 1         # Segments
    soma_single_compartment.Ra   = 110       # ohm·cm Axial Resistance
    soma_single_compartment.cm   = 1         # µF/cm² Membrane Capacitance

    # Leak channel
    soma_single_compartment.insert('pas')
    soma_single_compartment.g_pas = 0.0006    # S/cm²
    soma_single_compartment.e_pas = -62.5     # mV, should strongly define the resting potential of the neuron.

    # Hodgkin–Huxley channels only (spike.mod): Na and delayed-rectifier K. A-type K, Ca and Ca-activated K channels removed.
    soma_single_compartment.insert('spike')
    
    soma_single_compartment.gnabar_spike = 0.080     # S/cm² : Na
    soma_single_compartment.gkbar_spike  = 0.018     # S/cm² : delayed-rectifier K
    
    soma_single_compartment.ena = 40               # mV
    soma_single_compartment.ek  = -75              # mV
    return soma_single_compartment


def run_soma_single_compartment(soma_single_compartment, amp_pa, dur, delay, tstop):
    """Inject a current pulse (amp_pa in pA, dur in ms, starting at delay ms) into the compartment,
    run to tstop ms, and return the recordings as arrays:
    t, v, m, h, n (gating variables) and i (injected current, pA)."""
    # 1. Stimulus: a current clamp in the middle of the compartment
    clamp = h.IClamp(soma_single_compartment(0.5))             # 0.5 indicates middle of the compartment
    clamp.delay = delay                        # Sets the start of the square current injection
    clamp.dur   = dur                          # Sets the duration of the square current injection
    clamp.amp   = amp_pa / 1000                # IClamp works in nA

    # 2. Define what to record
    rec = {'t': h.Vector().record(h._ref_t),                    # Time vector
           'v': h.Vector().record(soma_single_compartment(0.5)._ref_v),         # Neuronal voltage response
           'm': h.Vector().record(soma_single_compartment(0.5)._ref_m_spike),   # Sodium activation, m
           'h': h.Vector().record(soma_single_compartment(0.5)._ref_h_spike),   # Sodium inactivation, h
           'n': h.Vector().record(soma_single_compartment(0.5)._ref_n_spike),   # Potassium, n
           'i': h.Vector().record(clamp._ref_i)}                # Stimulus Waveform

    # 3. Run: start at rest, then integrate to tstop
    h.dt      = 0.01                           # ms : time resolution of the simulation (fixed)
    h.tstop   = tstop                          # The timepoint where the simulation stops
    h.finitialize(-62.5)                       # resting potential (mV); gates start at steady state

    h.fcurrent()
    h.frecord_init()

    h.continuerun(tstop)                       # 'continuerun'-ning this simulation until the defined 'tstop'

    out = {k: np.array(x) for k, x in rec.items()}
    out['i'] *= 1000                           # nA -> pA
    return out # Output the stimulus waveform. Simulation data is stored in the global 'soma_single_compartment' variable


# Activity 4, 5
def load_rgc_model(morph_dir="neuron_morphology"):
    """Load the Sheasby & Fohlmeister (1999) RGC (ctt3219b) from morph_dir, flip it so the soma faces
    up, centre it on the soma, and insert the passive and extracellular (xtra) mechanisms.
    Does nothing if the model is already loaded."""
    if hasattr(h, 'soma'):
        print("Model already loaded.")
        return
    h.load_file(f"{morph_dir}/neuron_morphology_ctt3219b.asc")
    h.load_file(f"{morph_dir}/neuron_morphology_ctt3219b.hoc")

    def points(sec):
        return [(h.x3d(i, sec=sec), h.y3d(i, sec=sec), h.z3d(i, sec=sec), h.diam3d(i, sec=sec))
                for i in range(int(h.n3d(sec=sec)))]

    for sec in h.allsec():
        pts = points(sec)
        if len(pts) < 2:
            continue
        if sec == h.soma:
            pts = pts[::-1]
        sec.pt3dclear()
        for x, y, z, d in pts:
            sec.pt3dadd(x, y, -z, d)
    h.define_shape()

    # Centre the cell on the soma, and insert the mechanisms
    cx, cy, cz = np.mean([p[:3] for p in points(h.soma)], axis=0)
    for sec in h.allsec():
        sec.insert('extracellular')                    # extracellular field from a point electrode (xtra.mod)
        sec.insert('xtra')
        sec.insert('pas')
        pts = points(sec)
        if len(pts) < 2:
            continue
        sec.pt3dclear()
        for x, y, z, d in pts:
            sec.pt3dadd(x - cx, y - cy, z - cz, d)
        for seg in sec:                                # connect xtra to the extracellular mechanism
            h.setpointer(seg._ref_e_extracellular, 'ex', seg.xtra)
            h.setpointer(seg._ref_i_membrane, 'im', seg.xtra)

    h.define_shape()

def run_rgc(rgc, amp_pa, dur, delay, tstop, axon_diam=None):
    """Inject a pulse into the soma (amp_pa in pA, dur and delay in ms) and run to tstop ms, starting
    at rest. axon_diam (µm) changes the axon's diameter for this run only. Returns the results."""
    c = rgc['clamp']
    c.delay, c.dur, c.amp = delay, dur, amp_pa / 1000           # IClamp works in nA
    if axon_diam is not None:
        h.axon.diam = axon_diam
    h.dt, h.tstop, h.celsius = DT_FULL_CELL, tstop, 33
    h.finitialize(-62.5)                                        # resting potential (mV)
    for sec in h.allsec():                                      # resting calcium, so repeated runs agree
        if h.ismembrane('ca_ion', sec=sec):
            sec.cai = 0.0001
    h.fcurrent()
    h.frecord_init()
    h.continuerun(tstop)
    if axon_diam is not None:
        h.axon.diam = AXON_DIAM_DEFAULT                         # restore the model
    arr = lambda vec: np.array(vec)
    return dict(rgc=rgc, amp=amp_pa, dur=dur, delay=delay, axon_diam=axon_diam,
                t=arr(rgc['t']), i=arr(rgc['i']) * 1000, soma=arr(rgc['soma']), ais=arr(rgc['ais']),
                dend={l: arr(v) for l, v in rgc['dend'].items()},
                axon={l: arr(v) for l, v in rgc['axon'].items()})



## Activity 1: Single action potentials

In [ ]:
# Activity 1: a brief current pulse

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (< 5 ms) (Fill this in yourself!)
PULSE_DELAY = 10     # ms

T_STOP      = 40     # ms

SA_X        = 1      # multiplier on the neuron's surface area (only affects Activity 1)
#############################

soma_single_compartment = build_soma_single_compartment()
scale_surface_area(soma_single_compartment, SA_X)
r = run_soma_single_compartment(soma_single_compartment, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP)
show_single_compartment(r, soma_single_compartment, f"Surface area × {SA_X}: {PULSE_AMP} pA for {PULSE_DUR} ms")

## Activity 2: Tonic firing

In [ ]:
# Activity 2: a long current pulse

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (> 100 ms) (Fill this in yourself!)
PULSE_DELAY = 10     # ms

T_STOP      = 250    # ms
#############################

soma_single_compartment = build_soma_single_compartment()
r = run_soma_single_compartment(soma_single_compartment, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP)
show_single_compartment(r, soma_single_compartment, f"{PULSE_AMP} pA for {PULSE_DUR} ms")
report_tonic_firing(r, PULSE_DELAY, PULSE_DUR)

## Activity 3: Altering conductances

In [ ]:
# Activity 3: scale the Na and K conductances

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself!)
PULSE_DUR   = 0      # ms (Fill this in yourself!)
PULSE_DELAY = 10     # ms

T_STOP      = 40     # ms

NA_X        = 1      # multiplier on the Na conductance
K_X         = 1      # multiplier on the delayed-rectifier K conductance
#############################

soma_single_compartment = build_soma_single_compartment()
soma_single_compartment.gnabar_spike *= NA_X
soma_single_compartment.gkbar_spike  *= K_X
r = run_soma_single_compartment(soma_single_compartment, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP)
show_single_compartment(r, soma_single_compartment, f"gNa × {NA_X}, gK × {K_X}: {PULSE_AMP} pA for {PULSE_DUR} ms")
report_conductances(r, soma_single_compartment, PULSE_DELAY)

## Activity 4: Action potential generation and propagation
The full multi-compartment retinal ganglion cell. A brief pulse is injected into the soma, and the cell shows:
1. the standard figure (current, Vm at the soma, AIS, dendrite and axon, gates and conductances), with when the AIS and soma peak and how far the spike spreads
2. an interactive 3D dashboard of the whole cell
3. the same dashboard zoomed in on the soma and AIS, slowed down
4. how fast the spike travels along the axon (conduction velocity)

Find the threshold first: change `PULSE_AMP` and re-run until the cell fires.

In [ ]:
# Activity 4: action potential generation and propagation in the full cell

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself! Find the threshold for your pulse duration)
PULSE_DUR   = 0      # ms (Fill this in yourself!)
PULSE_DELAY = 5      # ms

T_STOP      = 30     # ms (long enough for the spike to reach the far end of the axon)
#############################

rgc = setup_rgc()                                                    # load the cell (once) and set up the recordings
res = run_rgc(rgc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP)        # inject the pulse into the soma and run
show_rgc(res)                                                        # 1. traces, gates and conductances
rgc_dashboard(res)                                                   # 2. 3D dashboard of the whole cell
rgc_zoom_dashboard(res)                                              # 3. zoomed in on the soma and AIS, slowed down
conduction_velocity(res)                                             # 4. conduction velocity along the axon

## Activity 5: Axonal conduction and axon diameter

In [ ]:
# Activity 5: conduction velocity for one axon diameter

####### PARAMETERS ##########
PULSE_AMP   = 0      # pA (Fill this in yourself! Use a value comfortably above threshold)
PULSE_DUR   = 0      # ms (Fill this in yourself!)
PULSE_DELAY = 5      # ms

T_STOP      = 30     # ms

AXON_DIAM   = 1.0    # µm (the model's axon is 1 µm)
#############################

rgc = setup_rgc()
res = run_rgc(rgc, PULSE_AMP, PULSE_DUR, PULSE_DELAY, T_STOP, axon_diam=AXON_DIAM)
show_axon_conduction(res)